# F2wLFSRGen — F2w-LFSR equidistribution

**F2wLFSR** (Panneton-L'Ecuyer 2004) lifts a Tausworthe-style LFSR
from $\mathbb{F}_2$ to $\mathrm{GF}(2^w)$, processing $w$ bits per
recurrence step instead of one. Parameters are the field defining
polynomial `modM`, the recurrence coefficients `coeff` (one element
of $\mathrm{GF}(2^w)$ per non-zero recurrence term), and the
matching position list `nocoeff`.

This notebook uses the first entry from Panneton-L'Ecuyer 2004
Table 1. See [generators/F2wLFSRGen.md](../../generators/F2wLFSRGen.md).


## Imports


In [ ]:
# stamp:auto-generated
from regpoly.core.generator import Generator
from regpoly.core.combination import Combination
from regpoly.core.transformation import Transformation
from regpoly.analyses.equidistribution_test import (
    EquidistributionTest,
    METHOD_MATRICIAL, METHOD_HARASE,
    METHOD_NOTPRIMITIVE, METHOD_SIMD_NOTPRIMITIVE,
)

INT_MAX = 2**31 - 1


## Construct the generator — _Panneton & L'Ecuyer (2004)_


In [ ]:
# F2w-LFSR, Panneton-L'Ecuyer 2004 — Table 1 row 1 (r=3, w=32).
gen = Generator.create("F2wLFSRGen", L=32,
    w=32, r=3, nb_terms=2, step=1, normal_basis=False,
    modM=0xCCB06F34,
    coeff=[0x30A2DEE7, 0x53782E5F],
    nocoeff=[1, 0])
print(gen.display())


## Wrap in a `Combination`


In [ ]:
# Wrap the generator in a single-component Combination. The
# Combination is the live object the search loop iterates and the
# equidistribution test consumes.
comb = Combination(J=1, Lmax=gen.L)
comb.components[0].add_gen(gen)
comb.reset()
print(f"k_g = {comb.k_g}, L = {comb.L}")


## Equidistribution test

F2w-LFSR with primitive `modM` and good coefficients is full-period; **Harase** applies.


In [ ]:
# Build the equidistribution test and run it. We cap `delta` at
# INT_MAX so nothing is rejected — we just want the score.
test = EquidistributionTest(
    L=gen.L,
    delta=[INT_MAX] * (gen.L + 1),
    mse=INT_MAX,
    method=METHOD_HARASE,
)
result = test.run(comb)

print(f"SE (Σ gaps)   = {result.se}")
print(f"verified      = {result.verified}")
print(f"first 10 gaps = {[result.ecart[i] for i in range(1, min(11, gen.L + 1))]}")


## Catalog entry

The published version of this parameter set lives in the REGPOLY catalog under `library_id = "f2w-lfsr-t1-r01"`. To load it programmatically without hard-coding parameters:

```python
from regpoly.library import Catalog
cat = Catalog('docs/library')
cat.load()
_, entry = cat.generator('f2w-lfsr-t1-r01')
# entry.components[0] carries the same params as constructed above
```
